In [16]:
%pip install numpy trimesh
import numpy as np
import trimesh
import matplotlib.pyplot as plt
from scipy.spatial import Delaunay
import time


Note: you may need to restart the kernel to use updated packages.


# Poisson Equation in 2D
$$-\Delta_\Gamma u=f \quad \text{(Laplace Beltrami)}$$


## Weak formulation
$u\in C^{2}(\Omega) \rightarrow u \in C^1(\Omega)$

$$$$

$$-\int_\Gamma v \Delta_\Gamma u \hspace{1mm} dS=\int_\Gamma fv \hspace{1mm} dS$$

$$$$

$$\int_\Gamma \nabla_\Gamma u \cdot \nabla_\Gamma v \hspace{1mm} dS - \int_{\partial \Gamma} \frac{\partial u}{\partial \nu} v \hspace{1mm} ds =\int_\Omega fv \hspace{1mm} dS \quad \text{(Divergence Theorem)}$$

# Mesh Solution

In a mesh our domain is partitioned into triangular meshes, where each vertex has its own $u$ value. Each point on the surface can then be approximated by an interpolation within the triangular face (element)

## Basis Function

We will utilize a simple lagrange basis which utilizes a quadratic triangle.


$$\Psi_i = \bigg\{ \begin{matrix} 1 & \text{vertex/midpoint i} \\ 0 & \text{not vertex/midpoint i}\end{matrix}$$
 
## Discretization

$$u_h(x,y)=\sum_{i}^NU_i\Psi_i(x,y)$$
$$$$
$$N = \text{total number of vertices}$$
$$U_i = \text{value of u at vertex i}$$
$$\Psi_i = \text{basis function of vertex i}$$

$$$$

 $u=u_h$ and $v=\sum_i \Psi_i \quad \text{(Galerkin Method)} $

$$$$
Ignoring boundary conditions, we get a system of equations involving a stiffness matrix (A) and a load vector (b)

$$$$

$$\int_\Gamma \left(\sum_i^N u_i\nabla_\Gamma \Psi_i\right) \left(\sum_j^N \nabla_\Gamma \Psi_j\right) d S =\int_\Gamma \sum_i^N \Psi_i f \hspace{1mm}
dS $$

$$$$

$$A_{i,j} = \int_{\Omega}\nabla_\Gamma \Psi_i \cdot \nabla_\Gamma \Psi_j \hspace{1mm} dS$$
$$b_{i} = \int_{\Gamma} \Psi_i f \hspace{1mm} dS$$

$$$$

$$AU=b, \quad U_i=u_i$$
$$$$
$$A\in \mathbb{R}^{N \times N},\quad b\in \mathbb{R}^{N},\quad U\in \mathbb{R}^{N}$$

# Elements

A quadratic triangle has 6 nodes, the three vertices $p_i,p_j,p_k$ and three midpoints $p_{jk},p_{ik},p_{ij}$ In our local coordinate system ($\xi$, $\eta$) we have:

- $p_i = (0,0)$
- $p_j = (0,1)$
- $p_k = (1,0)$
- $p_{jk} = (0.5,0.5)$
- $p_{ik} = (0.5,0)$
- $p_{ij} = (0,0.5)$

where the triangle is represented by $(\xi, \eta, 1-\xi-\eta)$

As a quadratic triangle, we want $u$ to be represented as a 2D quadratic polynomial across the element

$$u(\xi,\eta)=c_1+c_2\xi + c_3\eta + c_4 \xi^2 + c_5 \xi \eta + c_6 \eta^2$$

For each basis function $\Psi_m$ we want to make sure that $p_m$ is 1 and everything else is 0

For example for $\Psi_{jk}$:

$$\Psi_{jk}(\xi,\eta)=c_1+c_2\xi + c_3\eta + c_4 \xi^2 + c_5 \xi \eta + c_6 \eta^2$$
$$\Psi_{jk}(0,0) = 0, \Psi_{jk}(0,1) = 0, \Psi_{jk}(1,0) = 0, \Psi_{jk}(0.5,0.5)=1, \Psi_{jk}(0.5,0)=0, \Psi_{jk}(0.0.5)=0$$

consider the coefficient matrix $\mathbf{c}$

$$\begin{bmatrix}
1 & 0 & 0 & 0 & 0 & 0\\
1 & 1 & 0 & 1 & 0 & 0\\
1 & 0 & 1 & 0 & 0 & 1\\
1 & 0.5 & 0 & 0.25 & 0 & 0\\
1 & 0.5 & 0.5 & 0.25 & 0.25 & 0.25\\
1 & 0 & 0.5 & 0 & 0 & 0.25\\

\end{bmatrix}

\begin{bmatrix}
c_1 \\ c_2 \\ c_3 \\ c_4 \\ c_5 \\ c_6
\end{bmatrix}

=

\begin{bmatrix}
0 \\ 0 \\0  \\1 \\0 \\0
\end{bmatrix}

$$

Solving for $\mathbf{c}$ tells us that
$$\Psi_{jk}(\xi,\eta)=4\xi-4\xi^2-4\xi\eta$$

Doing this for all basis functions gives us that:

- $\Psi_i = \lambda(2\lambda-1) $
- $\Psi_j = \xi(2\xi-1)$
- $\Psi_k = \eta(2\eta-1)$
- $\Psi_{jk} = 4\xi\eta$
- $\Psi_{ik} = 4\eta\lambda$
- $\Psi_{ij} = 4\xi\lambda$

Where $\lambda=(1-\xi-\eta)$

Any point $(\mathbf{x})$ on the element would be the combination of the nodes ($\mathbf{x}_i$) and basis functions

$$\mathbf{x}(\xi,\eta)=\sum_{i=1}^6\Psi_i(\xi,\eta)\mathbf{x}_i$$

## Gradient

To determine the gradient at a point we can consider the gradient at the local coordinate system and then transform with a jacobian. (Covariant Basis)

$$J = \begin{bmatrix}
\mathbf{g}_1 & \mathbf{g}_2
\end{bmatrix}

=
\begin{bmatrix}
\frac{\partial x}{\partial \xi} & \frac{\partial x}{\partial \eta} \\
\frac{\partial y}{\partial \xi} & \frac{\partial y}{\partial \eta} \\
\frac{\partial z}{\partial \xi} & \frac{\partial z}{\partial \eta} \\
\end{bmatrix}
$$

Which can be computed explicity considering the basis functions

$$\mathbf{g}_1 = \sum_{i=1}^6 \frac{\partial \Psi_i}{\partial \xi} \mathbf{x}_i \quad \mathbf{g}_2 = \sum_{i=1}^6 \frac{\partial \Psi_i}{\partial \eta} \mathbf{x}_i$$

To make measurements we need to construct a metric tensor:

$$g=J^TJ = [g_{ij}] = \begin{bmatrix}

\mathbf{g}_1 \cdot \mathbf{g}_1 & \mathbf{g}_1 \cdot \mathbf{g}_2 \\
\mathbf{g}_2 \cdot \mathbf{g}_1 & \mathbf{g}_2 \cdot \mathbf{g}_2

\end{bmatrix}$$

The area element can be written as $dA = |\mathbf{g_1}\times \mathbf{g_2}| d\xi d\eta = \sqrt{\det(g)}d\xi d\eta$  

To project these derivatives back to physical space we need the inverse of the metric tensor (Contravariant Basis)

$$g^{-1} = [g^{ij}] = \frac{1}{\det(g)} \begin{bmatrix} g_{22} & -g_{12} \\ -g_{21} & g_{11}\end{bmatrix}$$

We can utilize this to compute the contravariant tangent vectors $\mathbf{g}^1$ and $\mathbf{g}^2$ These are dual vectors to the original tangent vectors ($\mathbf{g}^i \cdot \mathbf{g}_j = \delta_{ij}$, where $\delta_{ij}$ is the kronecker delta function)

$$\mathbf{g}^1 = g^{11} \mathbf{g}_1 + g^{12} \mathbf{g_2}$$
$$\mathbf{g}^2 = g^{21} \mathbf{g}_1 + g^{22} \mathbf{g_2}$$

Finally we can apply these contravariant dual vectors to the gradient of basis functions:

$$\nabla_\Gamma \Psi_i = \frac{\partial \Psi_i}{\partial \xi} \mathbf{g}^1 + \frac{\partial \Psi_i}{\partial \eta} \mathbf{g}^2$$

$$\nabla_\Gamma \Psi_i = \left(g^{11}\frac{\partial \Psi_i}{\partial \xi} + g^{12}\frac{\partial \Psi_i}{\partial \eta}\right)\mathbf{g}_1 +\left(g^{21}\frac{\partial \Psi_i}{\partial \xi} + g^{22}\frac{\partial \Psi_i}{\partial \eta}\right) \mathbf{g}_2$$

$$J^\dagger = g^{-1} J^T$$

$$\nabla_{\Gamma} \Psi_i = (J^\dagger)^T \begin{bmatrix} \frac{\partial N_i}{\partial \xi} \\ \frac{\partial N_i}{\partial \eta} \end{bmatrix}$$



In [2]:
class Element:
    def __init__(self, vertices):
        vertices = np.array(vertices)[:3]
        midpoints = np.array([
            (vertices[1] + vertices[2]) / 2,
            (vertices[0] + vertices[2]) / 2,
            (vertices[0] + vertices[1]) / 2  
        ])

        self.nodes = np.vstack((vertices, midpoints))
        self.nodes_local = [
            (0, 0),       # i
            (1, 0),       # j
            (0, 1),       # k
            (0.5, 0.5),   # m_jk
            (0, 0.5),     # m_ik
            (0.5, 0)      # m_ij
            ]

    def basis_functions(self,xi,eta):
        lam = 1 - xi - eta
        return [lam * (2 * lam - 1), xi * (2 * xi - 1), eta * (2 * eta - 1), 4 * xi * eta, 4 * eta * lam, 4 * xi * lam]
        
    def basis_function_gradients(self, xi, eta):
        lam = 1 - xi - eta
        return np.array([
        [-4 * lam + 1,       -4 * lam + 1],        # phi_i
        [ 4 * xi - 1,         0],                  # phi_j
        [ 0,                  4 * eta - 1],        # phi_k
        [ 4 * eta,            4 * xi],             # phi_jk
        [-4 * eta,            4 * lam - 4 * eta],  # phi_ik
        [ 4 * lam - 4 * xi,  -4 * xi],             # phi_ij
    ])
    
    def compute_jacobian(self, xi, eta):
        dpsi_dxi = self.basis_function_gradients(xi, eta)
        return self.nodes.T @ dpsi_dxi

    def compute_covariant(self,xi, eta):
        jacobian = self.compute_jacobian(xi, eta)
        return jacobian.T @ jacobian

    def compute_contravariant(self,xi, eta):
        return np.linalg.inv(self.compute_covariant(xi, eta))
    

    def pseudoinverse(self,xi, eta):
        return self.compute_contravariant(xi, eta) @ self.compute_jacobian(xi, eta).T

    def compute_gradients(self,xi, eta):

        gradients = np.zeros((6, 3))
        dpsi_dxi = self.basis_function_gradients(xi, eta)
        pinv_T = self.pseudoinverse(xi, eta).T         
        
        for i in range(6):
            gradients[i] = pinv_T @ dpsi_dxi[i]
            
        return gradients

    def compute_area(self,xi, eta):
        return np.sqrt(np.linalg.det(self.compute_covariant(xi, eta)))

    def local_to_global(self, local_coords):
        xi, eta = local_coords
        return np.array(self.basis_functions(xi, eta)) @ self.nodes


#  Local Stiffness Matrix

$$A_{i,j}^e=\int_{\Gamma_e}\nabla_\Gamma \Psi_i \cdot \nabla_\Gamma \Psi_j  dS$$

This integral is nonlinear so we will evaluate it with a gaussian quadrature, we will use a 7 point gaussian quadrature.


$$A_{i,j}^e=\sum_{k=1}^3\nabla_\Gamma \Psi_i(\xi_k,\eta_k) \cdot \nabla_\Gamma \Psi_j(\xi_k,\eta_k) \cdot w_k \cdot E_a$$

$$A^e=\begin{bmatrix}
A_{0,0} & A_{0,1} & A_{0,2} \\
A_{1,0} & A_{1,1} & A_{1,2} \\
A_{2,0} & A_{2,1} & A_{2,2} \\
\end{bmatrix}$$

$$$$

Note that these indices are local, they each represent a global index from $0...N$

# Global Stiffness Matrix
for each vertex add all the local entries to their corresponding global entry


In [3]:
def quadrature_points():
    a = 0.059715871789770
    b = 0.470142064105115
    c = 0.797426985353087
    d = 0.101286507323456

    w_center = 0.112500000000000
    w1 = 0.066197076394253
    w2 = 0.0629695902724135

    GP = np.array([
        [1/3, 1/3],

        [b, b],
        [a, b],
        [b, a],

        [d, d],
        [c, d],
        [d, c],
    ])

    weights = np.array([
        w_center,
        w1, w1, w1,
        w2, w2, w2
    ])

    return GP, weights

def update_global_stiffness(A,Vertices,face):
  element = Element(Vertices[face])
  GP, weights = quadrature_points()
  for i in range(6):
    for j in range(6):
      integral = 0
      for k in range(len(GP)):
        xi, eta = GP[k]
        weight = weights[k]
        grad_i,grad_j = element.compute_gradients(xi, eta)[i], element.compute_gradients(xi, eta)[j]
        integral +=   weight * grad_i.dot(grad_j) * element.compute_area(xi, eta)
      A[face[i],face[j]] += integral


def create_global_stiffness(A,Vertices,Faces):
  for face in Faces:
    update_global_stiffness(A,Vertices,face)


# Local Load Vector

$$b_i^e = \int_{\Gamma_e} f \Psi_i \hspace{1mm} dS$$

## Quadrature

We will utilize the same quadrature as the stiffness matrix
$$$$
$$b_i^e=\sum_{k=1}^3 f(\mathbf{x}(\xi_k,\eta_k)) \cdot \nabla_\Gamma \Psi_i(\xi_k,\eta_k) \cdot w_k \cdot E_a$$

# Global Load Vector
for each vertex add all the local entries to their corresponding global entry

In [4]:
def update_global_load(b,f,Vertices,face):
  element = Element(Vertices[face])
  GP, weights = quadrature_points()
  for i in range(6):
    integral = 0
    for k in range(len(GP)):
      xi, eta = GP[k]
      weight = weights[k]
      x,y,z = element.local_to_global((xi,eta))
      integral +=  weight * f(x,y,z) * element.basis_functions(xi, eta)[i] * element.compute_area(xi, eta)
    b[face[i]] += integral


def create_global_load(b,f,Vertices,Faces):
  for face in Faces:
    update_global_load(b,f,Vertices,face)



# Determining Boundary

If an edge is a boundary, then that edge should only appear in one element. To determine the edges simple find each unique edge. You can then determine the boudary vertices directly from the boundary edges.

$$$$

We need the boundary to apply the following boundary condition term:
$$$$
$$\int_{\partial \Gamma} \frac{\partial u}{\partial \nu} v \hspace{1mm} ds$$

In [5]:
def make_quadratic_mesh(Vertices, Faces):
    edge_to_midpoint = {}
    new_faces = []
    vertices_list = list(Vertices)

    for face in Faces:
        i, j, k = face
        edges = [(j, k), (i, k), (i, j)]
        midpoints_indices = []

        for edge in edges:
            sorted_edge = tuple(sorted(edge))
            if sorted_edge not in edge_to_midpoint:
                v1, v2 = sorted_edge
                midpoint = (vertices_list[v1] + vertices_list[v2]) / 2
                vertices_list.append(midpoint)
                edge_to_midpoint[sorted_edge] = len(vertices_list) - 1
            midpoints_indices.append(edge_to_midpoint[sorted_edge])

        new_faces.append([i, j, k, midpoints_indices[0], midpoints_indices[1], midpoints_indices[2]])

    return np.array(vertices_list), np.array(new_faces)


def get_Boundary(Faces):
    boundary_edges = []
    edge_count = {}
    boundary_vertices = set()

    # Step 1: Find boundary edges
    for face in Faces:
        corners = face[:3]
        edges = [
            tuple(sorted((corners[0], corners[1]))),
            tuple(sorted((corners[1], corners[2]))),
            tuple(sorted((corners[2], corners[0])))
        ]
        for edge in edges:
            edge_count[edge] = edge_count.get(edge, 0) + 1

    # Step 2: Identify structural boundary edges
    for edge, count in edge_count.items():
        if count == 1:
            boundary_edges.append(edge)
            boundary_vertices.add(edge[0])
            boundary_vertices.add(edge[1])

    # Step 3: Match quadratic midpoints to these boundary edges
    for face in Faces:
        i, j, k, m_jk, m_ik, m_ij = face
      
        if tuple(sorted((j, k))) in boundary_edges:
            boundary_vertices.add(m_jk)
        if tuple(sorted((k, i))) in boundary_edges:
            boundary_vertices.add(m_ik)
        if tuple(sorted((i, j))) in boundary_edges:
            boundary_vertices.add(m_ij)

    return list(boundary_vertices), boundary_edges



# Dirichlet Boundary Condition

$$u=g \quad \text{on} \hspace{3mm}\Gamma_d$$

This means for every boundary ($d$), $U_d=g(p_d)=g_d$

We need to reset all the boundary values and apply the boundary value term in the weak formulation
$$$$

1. Zero out the row and column of boundary $A_{i,d} = A_{d,i}= 0$

2. Set diagonal entry to one $A_{d,d}=1$

3. set load vector entry to g $F_d=g_d$

$$$$

These steps ensure that $U_d = g_d$

In [6]:
def apply_dirichlet(A, b, Vertices, boundary_vertices, g):
    for d in boundary_vertices:
        x, y, z = Vertices[d]
        gd = g(x, y, z)

        # adjust RHS for all equations
        b[:] -= A[:, d] * gd

        # zero row and column
        A[d, :] = 0.0
        A[:, d] = 0.0

        # impose value
        A[d, d] = 1.0
        b[d] = gd

# Testing

In order to test the validity of our FEM we will consider comparing it to known solutions on the plane.

$$u(x,y)=(x^2-1)(y^2-1) \in \Gamma [-1,1] \times [-1,1]$$
$$-\Delta u = -2(x^2+y^2-2)$$
$$u(\Gamma_d) = 0 \Rightarrow g=0$$

In [7]:
%%script false --no-raise-error

def quad_integral(p, q):
    total = 0.0
    GP, weights = quadrature_points()
    for (GP, w) in zip(GP, weights):
        total += w * (GP[0]**p) * (GP[1]**q)
    return total

print(quad_integral(0, 0))  # should be 0.5
print(quad_integral(1, 0))  # should be 1/6
print(quad_integral(0, 1))  # should be 1/6
print(quad_integral(2, 0))  # should be 1/12
print(quad_integral(1, 1))  # should be 1/24

Couldn't find program: 'false'


In [8]:
%%script false --no-raise-error

X = np.linspace(-1,1,3)
Y = np.linspace(-1,1,3)

domain = np.array([[x, y] for x in X for y in Y])
tri = Delaunay(domain)

faces = tri.simplices
vertices = np.array([[x,y,0] for x in X for y in Y])
mesh  = trimesh.Trimesh(vertices,faces)

for i in range(5):
  mesh = mesh.subdivide()

#upgrade to quadratic mesh
Vertices,Faces = make_quadratic_mesh(mesh.vertices,mesh.faces)

#source term
def f(x,y,z):
  return -2*(x**2+y**2-2)

N = len(Vertices)

#Initialize System
A = np.zeros((N,N))
b = np.zeros(N)

#Load System
create_global_stiffness(A,Vertices,Faces)
create_global_load(b,f,Vertices,Faces)

#Get Bounds
boundary_vertices,boundary_edges = get_Boundary(Faces)


#Define Boundary Conditions
def g(x,y,z):
  return 0

def h(x,y,z):
  return 0

def alpha(x,y,z):
  return 1

def r(x,y,z):
  return 0

#Apply Boundary Conditions

apply_dirichlet(A,b,Vertices,boundary_vertices,g)
#apply_neumann(A,b,Vertices,boundary_edges,h)
#apply_robin(A,b,Vertices,boundary_edges,alpha,r)

#Solve
U = np.linalg.solve(A,b)

#Save Results
File = open('./solutions/planar-quad-O2/5.csv','w')
File.write('x,y,u\n')

print(len(U))

boundary_set = set(boundary_vertices)

for i in range(N):
    (x, y, z) = Vertices[i]
    if i in boundary_set:
        File.write(f"{x}, {y}, {U[i]}, boundary\n")
    else:
        File.write(f"{x}, {y}, {U[i]}\n")

File.close()


Couldn't find program: 'false'


# Test 2

In [ ]:
X = np.linspace(-1,1,3)
Y = np.linspace(-1,1,3)

domain = np.array([[x, y] for x in X for y in Y])
tri = Delaunay(domain)

faces = tri.simplices
vertices = np.array([[x,y,0] for x in X for y in Y])
mesh  = trimesh.Trimesh(vertices,faces)

for i in range(5):
    mesh = mesh.subdivide()
    
    current_vertices = mesh.vertices
    updated_vertices = []
    for p in current_vertices:
        x, y = p[0], p[1]
        z = x**2 + y**2  # Force the vertex onto the paraboloid geometry
        updated_vertices.append([x, y, z])
    
    # Re-assign the snapped vertices back to the mesh object
    mesh.vertices = np.array(updated_vertices)

quad_vertices, quad_faces = make_quadratic_mesh(mesh.vertices, mesh.faces)

# Proceed to assign your final Vertices and Faces for the FEM loops
Vertices = quad_vertices
Faces = quad_faces

#source term
def f(x, y, z):
    W_sq = 1.0 + 4.0 * x**2 + 4.0 * y**2
    term1 = -2.0 * (y**2 - 1.0) - 2.0 * (x**2 - 1.0)
    term2 = (8.0 / W_sq) * (x**2 * (y**2 - 1.0) + y**2 * (x**2 - 1.0))
    return (term1 + term2) / W_sq

N = len(Vertices)

#Initialize System
A = np.zeros((N,N))
b = np.zeros(N)

start = time.perf_counter()

#Load System
create_global_stiffness(A,Vertices,Faces)
create_global_load(b,f,Vertices,Faces)

#Get Bounds
boundary_vertices,boundary_edges = get_Boundary(Faces)


#Define Boundary Conditions
def g(x,y,z):
  return 0

def h(x,y,z):
  return 0

def alpha(x,y,z):
  return 1

def r(x,y,z):
  return 0

#Apply Boundary Conditions

apply_dirichlet(A,b,Vertices,boundary_vertices,g)
#apply_neumann(A,b,Vertices,boundary_edges,h)
#apply_robin(A,b,Vertices,boundary_edges,alpha,r)

#Solve
U = np.linalg.solve(A,b)

stop = time.perf_counter()
print(f"Time taken to solve the system: {stop - start:.6f} seconds")

#Save Results
#File = open('./solutions/parab-quad-O2/5.csv','w')
# File.write('x,y,u\n')

print(len(U))

# boundary_set = set(boundary_vertices)

# for i in range(N):
#     (x, y, z) = Vertices[i]
#     if i in boundary_set:
#         File.write(f"{x}, {y}, {z}, {U[i]}, boundary\n")
#     else:
#         File.write(f"{x}, {y}, {z}, {U[i]}\n")

# File.close()


IndexError: index 3 is out of bounds for axis 0 with size 3